# Phase 6: Experiments and the Critical Adoption Share A*

Goal: sweep four parameters (mu, phi, forecast window w, and AR order p) and for each combination locate the critical adoption share A* at which the proposal's mechanism has bitten. Produce headline heatmaps for the AR(1) baseline, line plots and difference heatmaps that compare higher AR orders, and a representative erosion-path figure. CLAUDE.md describes phase 6 as the compact headline result of the project.

Reference: section 4.2 of the proposal for A* and the threshold metrics, section 3.5 for stochastic diffusion, and phase 4 for the residual-vs-realised R^2 distinction.

**Window vs order.** Two distinct knobs that the sweep separates:

- `w` is the rolling estimation window length. Adopters fit the public AR model on the most recent `w` realised returns. Larger `w` smooths the parameter estimate.
- `p` is the AR order. AR(1) is the proposal's baseline rule (only the first lag enters the forecast). Higher `p` lets the rule pick up multi-lag structure. AR(5) and AR(10) are robustness or extension checks: if A* is qualitatively similar at higher `p`, the headline mechanism does not depend on the AR(1) specification.

**A note on the threshold.** The proposal defines A* as the smallest A at which the rolling out-of-sample R^2 crosses zero, or net profit crosses zero. In this cost-free baseline the residual R^2 falls but stays positive (typically ~0.025 even at full adoption) and adopter mean profit stays positive (it grows with adoption, by analytical inspection). So an absolute zero crossing does not fire. The cells below use **relative** thresholds instead: A*_R2 is the smallest A at which the rolling residual R^2 has fallen to half its baseline value, and A*_phi is the smallest A at which the rolling effective phi has grown to 1.5x its baseline. Once transaction costs land in phase 7 the absolute zero-crossing definition becomes operational and we swap back.

**Three correctness fences.** (a) Crossings are searched only at indices past the baseline window, so rolling-metric noise inside the low-adoption span cannot satisfy the threshold prematurely. (b) Any cell whose hit rate (fraction of seeds in which a crossing actually happened) falls below 50% is reported as NaN to avoid a noisy minority of seeds setting the cell. (c) Each AR(p) fit needs more observations than coefficients, so combinations with `w <= p` are skipped and stored as NaN; with the current grids all 4 x 3 = 12 (p, w) combinations are valid.

**Paired seeds across p.** For the same `(w, phi, mu, seed_index)`, every AR order uses the same input seed. The simulator splits that rng into market and adoption sub-rngs deterministically, so paired runs share the same exogenous shock stream and adoption transitions; the only difference across p is adopter behaviour through the forecast. Differences in A* across p therefore come from the AR order, not from different Monte Carlo draws.

**Speed.** The seed loop is parallelised with `joblib.Parallel(n_jobs=-1)`. The simulator is pure and deterministic given seed, so it parallelises for free. The full 4 x 3 x 6 x 5 x 10 = 3600-run sweep finishes in ~15 minutes on 16 cores.

In [ ]:
# Common run parameters.
N = 200
T = 8000
sigma_news = 0.01
sigma_q = 1.0
risk_scale = 0.001
q_cap = 0.05
eval_window = 1000
adoption_pi = 3e-4   # slow stochastic diffusion: A ramps from 0 to ~0.87 over the run

# Sweep grid (lists; converted to numpy in the run cell).
mu_grid = [0.025, 0.05, 0.075, 0.10, 0.15]
phi_grid = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
w_grid = [100, 250, 500]
# AR order grid. p = 1 is the proposal's baseline; the higher orders are
# robustness extensions. Combinations with w <= p are skipped.
forecast_p_grid = [1, 2, 5, 10]

# Number of seeds per (p, w, phi, mu) cell. With joblib parallelism the
# 4 x 3 x 6 x 5 x 10 = 3600 runs finish in ~15 min on 16 cores.
num_seeds_phase6 = 10
base_seed = 2000

# Relative thresholds for A*. The R^2 threshold is hit when the rolling
# residual R^2 has dropped to half its baseline; the phi threshold is
# hit when the rolling effective phi has grown by 50%.
r2_threshold_factor = 0.5
phi_threshold_factor = 1.5

# Cells with hit rate below this are reported as NaN: a noisy minority of
# seeds crossing the threshold should not pretend to set the cell.
min_hit_rate = 0.5

# Length of the low-adoption baseline window after the rolling-metric warmup.
baseline_span = 1500

fig_dir = "../results/figures"
data_dir = "../results/data"

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

from reflexive_market import simulate
from reflexive_market.metrics import rolling_oos_r2, rolling_phi

In [ ]:
def find_a_star(adoption_share, metric, threshold, direction, start_idx=0):
    """Smallest adoption share at indices >= start_idx where metric crosses threshold.

    direction == 'below': smallest A where metric < threshold (R^2 erosion).
    direction == 'above': smallest A where metric > threshold (phi growth).
    Restricting to indices past the baseline window keeps rolling-metric noise
    inside the low-adoption span from satisfying the threshold before adoption
    has had a chance to erode anything.
    """
    A = adoption_share[start_idx:]
    M = metric[start_idx:]
    if direction == "below":
        crossed = np.isfinite(M) & (M < threshold)
    else:
        crossed = np.isfinite(M) & (M > threshold)
    if not np.any(crossed):
        return np.nan
    return float(A[crossed].min())


def run_seed(seed_s, p, fw, adoption_start_t_w, baseline_lo_w, baseline_hi_w,
             mu_v, phi_v):
    """One simulator run + threshold detection for one seed and one (p, w, phi, mu) cell."""
    rng_s = np.random.default_rng(seed_s)
    out = simulate.run(
        T=T, N=N, mu=mu_v, phi=phi_v,
        sigma_news=sigma_news, sigma_q=sigma_q,
        rng=rng_s,
        forecast_window=fw, forecast_p=p,
        risk_scale=risk_scale, q_cap=q_cap,
        adoption_pi=adoption_pi, adoption_delta=0.0,
        adoption_start_t=adoption_start_t_w,
    )
    A = out["adoption_share"]
    residual = out["returns"] - mu_v * out["demand"]
    R2 = rolling_oos_r2(residual, out["forecasts"], eval_window)
    phi_eff = rolling_phi(out["returns"], eval_window)

    base_R2 = float(np.nanmean(R2[baseline_lo_w:baseline_hi_w]))
    base_phi = float(np.nanmean(phi_eff[baseline_lo_w:baseline_hi_w]))

    a_star_R2_s = (
        find_a_star(A, R2, r2_threshold_factor * base_R2, "below",
                    start_idx=baseline_hi_w)
        if base_R2 > 0 else np.nan
    )
    a_star_phi_s = (
        find_a_star(A, phi_eff, phi_threshold_factor * base_phi, "above",
                    start_idx=baseline_hi_w)
        if base_phi > 0 else np.nan
    )
    return base_R2, base_phi, a_star_R2_s, a_star_phi_s

In [ ]:
mu_arr = np.array(mu_grid)
phi_arr = np.array(phi_grid)
w_arr = np.array(w_grid, dtype=int)
p_arr = np.array(forecast_p_grid, dtype=int)
n_mu = mu_arr.size
n_phi = phi_arr.size
n_w = w_arr.size
n_p = p_arr.size

# Build a flat job list of (ip, k, i, j, seed_s, p, fw, adoption_start_t_w,
# baseline_lo_w, baseline_hi_w, mu_v, phi_v) tuples. Flat is the only
# shape that lets joblib fully saturate the workers.
#
# The seed formula deliberately omits the AR-order index ip, so for a
# given (w, phi, mu, seed_index) every AR order reuses the same input
# seed. The simulator splits that rng into market and adoption sub-rngs
# deterministically, so paired runs share the same shock stream and the
# only difference across p is adopter behaviour through the forecast.
#
# Adoption is held off until the baseline window has finished, so the
# baseline R^2/phi are measured with A = 0 across their full span and
# find_a_star starts looking for crossings from A = 0.
#
# Combinations with fw <= p are skipped (the AR(p) fit needs more
# observations than coefficients); the corresponding grid entries stay
# at their initialised NaN values and the heatmap shows them blank.
jobs = []
skipped = 0
for ip, p in enumerate(p_arr):
    p_int = int(p)
    for k, w_v in enumerate(w_arr):
        fw = int(w_v)
        if fw <= p_int:
            skipped += n_phi * n_mu * num_seeds_phase6
            continue
        baseline_lo_w = fw + eval_window
        baseline_hi_w = baseline_lo_w + baseline_span
        adoption_start_t_w = baseline_hi_w
        for i, phi_v in enumerate(phi_arr):
            for j, mu_v in enumerate(mu_arr):
                for s in range(num_seeds_phase6):
                    seed_s = (base_seed
                              + 100000 * k
                              + 1000 * (i * n_mu + j)
                              + s)
                    jobs.append((ip, k, i, j, seed_s, p_int, fw,
                                  adoption_start_t_w, baseline_lo_w, baseline_hi_w,
                                  float(mu_v), float(phi_v)))

print(
    f"Submitting {len(jobs)} simulator runs across {n_p}*{n_w}*{n_phi}*{n_mu} = "
    f"{n_p*n_w*n_phi*n_mu} cells, {num_seeds_phase6} seeds each "
    f"({skipped} skipped from w <= p)"
)

raw = Parallel(n_jobs=-1, verbose=5)(
    delayed(run_seed)(seed_s, p, fw, adoption_start_t_w, baseline_lo_w,
                       baseline_hi_w, mu_v, phi_v)
    for (_, _, _, _, seed_s, p, fw, adoption_start_t_w, baseline_lo_w,
         baseline_hi_w, mu_v, phi_v) in jobs
)

In [ ]:
# Aggregate per (ip, k, i, j) cell. Arrays are 4D: (p, w, phi, mu).
a_star_R2 = np.full((n_p, n_w, n_phi, n_mu), np.nan)
a_star_phi = np.full((n_p, n_w, n_phi, n_mu), np.nan)
hit_rate_R2 = np.zeros((n_p, n_w, n_phi, n_mu))
hit_rate_phi = np.zeros((n_p, n_w, n_phi, n_mu))
baseline_R2_grid = np.full((n_p, n_w, n_phi, n_mu), np.nan)
baseline_phi_grid = np.full((n_p, n_w, n_phi, n_mu), np.nan)

buckets = {}
for (ip, k, i, j, *_), result in zip(jobs, raw):
    key = (ip, k, i, j)
    buckets.setdefault(key, []).append(result)

for (ip, k, i, j), seeds in buckets.items():
    base_R2s = np.array([r[0] for r in seeds])
    base_phis = np.array([r[1] for r in seeds])
    a_R2s = np.array([r[2] for r in seeds])
    a_phis = np.array([r[3] for r in seeds])
    n_seeds = len(seeds)

    baseline_R2_grid[ip, k, i, j] = float(np.nanmean(base_R2s))
    baseline_phi_grid[ip, k, i, j] = float(np.nanmean(base_phis))

    # Hit rate denominator is the total number of seeds, not the number
    # with positive baseline. Otherwise a cell where most seeds have a
    # noisy non-positive baseline (so a_R2_s is set to NaN inside
    # run_seed) silently passes the filter when one outlier seed crosses.
    crossed_R2 = np.isfinite(a_R2s)
    hit_rate_R2[ip, k, i, j] = crossed_R2.sum() / n_seeds
    if hit_rate_R2[ip, k, i, j] >= min_hit_rate:
        a_star_R2[ip, k, i, j] = float(np.mean(a_R2s[crossed_R2]))

    crossed_phi = np.isfinite(a_phis)
    hit_rate_phi[ip, k, i, j] = crossed_phi.sum() / n_seeds
    if hit_rate_phi[ip, k, i, j] >= min_hit_rate:
        a_star_phi[ip, k, i, j] = float(np.mean(a_phis[crossed_phi]))

print(f"Sweep complete: {n_p}x{n_w}x{n_phi}x{n_mu} cells, {num_seeds_phase6} seeds each.")

In [ ]:
# Print one summary block per (p, w) panel. The AR(1) baseline reads first.
for ip, p in enumerate(p_arr):
    for k, w_v in enumerate(w_arr):
        print(f"\n--- p = {int(p)}, w = {int(w_v)} ---")
        header = (
            f"{'phi':>6} | {'mu':>8} | {'A*_R2':>8} | {'hitR2':>6} | "
            f"{'A*_phi':>8} | {'hitphi':>7} | {'baseR2':>9} | {'basephi':>9}"
        )
        print(header)
        print("-" * len(header))
        for i, phi_v in enumerate(phi_arr):
            for j, mu_v in enumerate(mu_arr):
                print(
                    f"{phi_v:>6.2f} | {mu_v:>8.3f} | "
                    f"{a_star_R2[ip, k, i, j]:>8.3f} | {hit_rate_R2[ip, k, i, j]:>6.2f} | "
                    f"{a_star_phi[ip, k, i, j]:>8.3f} | {hit_rate_phi[ip, k, i, j]:>7.2f} | "
                    f"{baseline_R2_grid[ip, k, i, j]:>9.4f} | {baseline_phi_grid[ip, k, i, j]:>9.4f}"
                )

## Headline heatmaps for AR(1)

These two figures are the proposal's baseline result: A* across (mu, phi) for each forecast window length, fixing the AR order at the proposal's default `p = 1`. They reproduce the original phase 6 view exactly. The next sections then add AR order p as a second sweep dimension.

In [ ]:
def plot_heatmap_row(values_3d, hit_3d, fname, suptitle):
    """One row of (mu, phi) heatmaps across the w axis. values_3d has shape (n_w, n_phi, n_mu)."""
    fig, axes = plt.subplots(1, n_w, figsize=(6 * n_w, 5), sharey=True)
    for k, w_v in enumerate(w_arr):
        ax = axes[k] if n_w > 1 else axes
        im = ax.imshow(values_3d[k], origin="lower", cmap="viridis_r",
                        vmin=0.0, vmax=1.0, aspect="auto")
        ax.set_xticks(range(n_mu))
        ax.set_xticklabels([f"{m:g}" for m in mu_arr])
        if k == 0:
            ax.set_yticks(range(n_phi))
            ax.set_yticklabels([f"{p:.2f}" for p in phi_arr])
            ax.set_ylabel("phi (input AR coefficient)")
        ax.set_xlabel("mu (price-impact coefficient)")
        ax.set_title(f"w = {int(w_v)}")
        for i in range(n_phi):
            for j in range(n_mu):
                v = values_3d[k, i, j]
                if not np.isfinite(v):
                    hr = hit_3d[k, i, j]
                    text = f"hit {hr:.1f}"
                    color = "black"
                else:
                    text = f"{v:.2f}"
                    color = "white" if v < 0.5 else "black"
                ax.text(j, i, text, ha="center", va="center", color=color, fontsize=8)
        fig.colorbar(im, ax=ax, label="A*")
    fig.suptitle(suptitle, y=1.02)
    fig.tight_layout()
    fig.savefig(f"{fig_dir}/{fname}", dpi=150, bbox_inches="tight")
    plt.show()


# AR(1) is index 0 of the p axis.
plot_heatmap_row(
    a_star_R2[0], hit_rate_R2[0],
    "phase_06_a_star_R2_heatmap.png",
    f"A* at residual R^2 = {r2_threshold_factor} x baseline, AR(1), hit rate >= {min_hit_rate} (over {num_seeds_phase6} seeds)",
)

In [ ]:
plot_heatmap_row(
    a_star_phi[0], hit_rate_phi[0],
    "phase_06_a_star_phi_heatmap.png",
    f"A* at effective phi = {phi_threshold_factor} x baseline, AR(1), hit rate >= {min_hit_rate} (over {num_seeds_phase6} seeds)",
)

## AR order sensitivity: A* across p

Rows are AR orders (`p = 1, 2, 5, 10`), columns are forecast windows (`w = 100, 250, 500`). The first row reproduces the AR(1) baseline above. If the residual-R^2 and effective-phi gradients hold qualitatively across rows, the headline mechanism is robust to the AR order specification.

In [ ]:
def plot_heatmap_grid(values_4d, hit_4d, suptitle, fname):
    """4D heatmap grid: rows over p, columns over w. Each cell is (mu, phi)."""
    fig, axes = plt.subplots(n_p, n_w, figsize=(5 * n_w, 4 * n_p), sharey=True)
    if n_p == 1 and n_w == 1:
        axes = np.array([[axes]])
    elif n_p == 1:
        axes = axes[np.newaxis, :]
    elif n_w == 1:
        axes = axes[:, np.newaxis]
    for ip, p in enumerate(p_arr):
        for k, w_v in enumerate(w_arr):
            ax = axes[ip, k]
            data = values_4d[ip, k]
            hit = hit_4d[ip, k]
            im = ax.imshow(data, origin="lower", cmap="viridis_r",
                            vmin=0.0, vmax=1.0, aspect="auto")
            ax.set_xticks(range(n_mu))
            ax.set_xticklabels([f"{m:g}" for m in mu_arr])
            if k == 0:
                ax.set_yticks(range(n_phi))
                ax.set_yticklabels([f"{ph:.2f}" for ph in phi_arr])
                ax.set_ylabel(f"p = {int(p)}\nphi")
            if ip == n_p - 1:
                ax.set_xlabel("mu")
            ax.set_title(f"p = {int(p)}, w = {int(w_v)}")
            for i in range(n_phi):
                for j in range(n_mu):
                    v = data[i, j]
                    if not np.isfinite(v):
                        text = f"hit {hit[i, j]:.1f}"
                        color = "black"
                    else:
                        text = f"{v:.2f}"
                        color = "white" if v < 0.5 else "black"
                    ax.text(j, i, text, ha="center", va="center", color=color, fontsize=7)
    fig.suptitle(suptitle, y=1.005)
    fig.tight_layout()
    fig.savefig(f"{fig_dir}/{fname}", dpi=150, bbox_inches="tight")
    plt.show()


plot_heatmap_grid(
    a_star_R2, hit_rate_R2,
    f"A* at residual R^2 = {r2_threshold_factor} x baseline, by AR order p and window w",
    "phase_06_a_star_r2_by_p.png",
)

In [ ]:
plot_heatmap_grid(
    a_star_phi, hit_rate_phi,
    f"A* at effective phi = {phi_threshold_factor} x baseline, by AR order p and window w",
    "phase_06_a_star_phi_by_p.png",
)

## Figure A: Baseline forecast performance by AR order

Before adoption fires, how does the rolling residual R^2 look as a function of AR order? Aggregating the baseline R^2 across the (phi, mu) grid for each (p, w) tells us whether higher-order AR starts the run with a better or worse forecast. A higher baseline R^2 means the rule begins with more predictive power to lose.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
p_x = np.arange(n_p)
offsets = np.linspace(-0.15, 0.15, n_w)
for k, w_v in enumerate(w_arr):
    means = np.full(n_p, np.nan)
    stds = np.full(n_p, np.nan)
    for ip, p in enumerate(p_arr):
        cells = baseline_R2_grid[ip, k]
        finite = cells[np.isfinite(cells)]
        if finite.size > 0:
            means[ip] = finite.mean()
            stds[ip] = finite.std()
    ax.errorbar(p_x + offsets[k], means, yerr=stds, marker="o",
                 capsize=3, label=f"w = {int(w_v)}")
ax.axhline(0.0, color="k", linewidth=0.5)
ax.set_xticks(p_x)
ax.set_xticklabels([str(int(p)) for p in p_arr])
ax.set_xlabel("AR order p")
ax.set_ylabel("Baseline residual R^2 (mean +/- 1 SD across (phi, mu) grid)")
ax.set_title("Baseline forecast performance by AR order, before adoption fires")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_06_baseline_r2_by_p.png", dpi=150)
plt.show()

## Figure B: A*_R2 vs AR order for three representative cells

How does the critical adoption share move as we change the AR order, at fixed (phi, mu)? Three illustrative cells:

- **Weak signal, weak feedback:** `phi = 0.10, mu = 0.025`
- **Middle:** `phi = 0.20, mu = 0.075`
- **Strong signal, strong feedback:** `phi = 0.30, mu = 0.15`

Each panel has one line per forecast window w. Missing points (visible gaps in the line) are cells where the hit-rate filter rejected the cell because fewer than 50% of seeds produced a crossing.

In [ ]:
rep_cells = [
    ("weak signal, weak feedback", 0.10, 0.025),
    ("middle", 0.20, 0.075),
    ("strong signal, strong feedback", 0.30, 0.15),
]

fig, axes = plt.subplots(1, len(rep_cells), figsize=(5 * len(rep_cells), 4), sharey=True)
for ax_idx, (label, phi_v, mu_v) in enumerate(rep_cells):
    ax = axes[ax_idx]
    i = int(np.argmin(np.abs(phi_arr - phi_v)))
    j = int(np.argmin(np.abs(mu_arr - mu_v)))
    for k, w_v in enumerate(w_arr):
        ys = a_star_R2[:, k, i, j]
        ax.plot(p_x, ys, marker="o", label=f"w = {int(w_v)}")
    ax.set_xticks(p_x)
    ax.set_xticklabels([str(int(p)) for p in p_arr])
    ax.set_xlabel("AR order p")
    if ax_idx == 0:
        ax.set_ylabel("A*_R2")
    ax.set_title(f"{label}\nphi = {phi_v}, mu = {mu_v}")
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("A*_R2 vs AR order p at three representative (phi, mu) cells", y=1.02)
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_06_a_star_r2_lines_by_p.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure C: Residual R^2 erosion curve vs adoption share by AR order

Pick one representative cell (`phi = 0.25, mu = 0.05, w = 250`) and one paired seed, run the simulator at each AR order, then plot the rolling residual R^2 against the adoption share. With paired seeds the exogenous shock stream and adoption transitions are identical across p (the simulator's adoption rng is split off deterministically from the input rng), so the only difference between the lines is adopter behaviour.

In [ ]:
line_curve_phi = 0.25
line_curve_mu = 0.05
line_curve_w = 250
line_curve_seed = base_seed   # any paired seed works; reuse the canonical one

line_curve_baseline_lo = line_curve_w + eval_window
line_curve_baseline_hi = line_curve_baseline_lo + baseline_span
line_curve_adoption_start = line_curve_baseline_hi

line_curve_residual_r2 = np.full((n_p, T), np.nan)
line_curve_phi_trace = np.full((n_p, T), np.nan)
line_curve_adoption_share = None   # shared across p when seeds are paired
for ip, p in enumerate(p_arr):
    rng_c = np.random.default_rng(line_curve_seed)
    out = simulate.run(
        T=T, N=N, mu=line_curve_mu, phi=line_curve_phi,
        sigma_news=sigma_news, sigma_q=sigma_q,
        rng=rng_c,
        forecast_window=line_curve_w, forecast_p=int(p),
        risk_scale=risk_scale, q_cap=q_cap,
        adoption_pi=adoption_pi, adoption_delta=0.0,
        adoption_start_t=line_curve_adoption_start,
    )
    if line_curve_adoption_share is None:
        line_curve_adoption_share = out["adoption_share"].copy()
    residual = out["returns"] - line_curve_mu * out["demand"]
    line_curve_residual_r2[ip] = rolling_oos_r2(residual, out["forecasts"], eval_window)
    line_curve_phi_trace[ip] = rolling_phi(out["returns"], eval_window)

# AR(1) baseline R^2 for the half-baseline reference line.
line_curve_baseline_ar1 = float(
    np.nanmean(line_curve_residual_r2[0, line_curve_baseline_lo:line_curve_baseline_hi])
)

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.viridis(np.linspace(0.0, 0.85, n_p))
for ip, p in enumerate(p_arr):
    r2 = line_curve_residual_r2[ip]
    mask = np.isfinite(r2)
    ax.plot(line_curve_adoption_share[mask], r2[mask], color=colors[ip],
             linewidth=1.5, label=f"p = {int(p)}")
ax.axhline(0.0, color="k", linewidth=0.5)
ax.axhline(
    r2_threshold_factor * line_curve_baseline_ar1,
    color="gray", linewidth=0.8, linestyle="--",
    label=f"{r2_threshold_factor} x AR(1) baseline",
)
ax.set_xlabel("Adoption share")
ax.set_ylabel("Rolling residual OOS R^2")
ax.set_title(
    f"Residual R^2 vs adoption by AR order p (phi={line_curve_phi}, mu={line_curve_mu}, "
    f"w={line_curve_w}, paired seed)"
)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_06_residual_r2_vs_adoption_by_p.png", dpi=150)
plt.show()

## Figure D: Delta from AR(1) heatmap

For each higher-order AR model, the difference `A*_R2(AR(p)) - A*_R2(AR(1))` across the (mu, phi) grid. Positive values (red) mean AR(p) reaches the relative R^2 threshold at a *higher* adoption share than AR(1) does (the higher-order rule is more robust to adoption pressure). Negative values (blue) mean AR(p) erodes earlier. Cells where either AR(1) or AR(p) fails the hit-rate filter are NaN and shown white. This is the clearest robustness summary: a heatmap close to zero everywhere means the headline mechanism is insensitive to AR order.

In [ ]:
delta_a_star_r2 = a_star_R2 - a_star_R2[0:1]   # broadcast subtract AR(1) row from all
# Choose a symmetric color range from observed deltas (excluding p=1 which is zero).
abs_max_finite = np.nanmax(np.abs(delta_a_star_r2[1:]))
abs_max = float(abs_max_finite) if np.isfinite(abs_max_finite) else 0.5
vlim = max(abs_max, 0.05)

higher_p_indices = list(range(1, n_p))
fig, axes = plt.subplots(len(higher_p_indices), n_w,
                          figsize=(5 * n_w, 4 * len(higher_p_indices)),
                          sharey=True)
if len(higher_p_indices) == 1:
    axes = axes[np.newaxis, :]
for row_idx, ip in enumerate(higher_p_indices):
    for k, w_v in enumerate(w_arr):
        ax = axes[row_idx, k]
        data = delta_a_star_r2[ip, k]
        im = ax.imshow(data, origin="lower", cmap="RdBu_r",
                        vmin=-vlim, vmax=vlim, aspect="auto")
        ax.set_xticks(range(n_mu))
        ax.set_xticklabels([f"{m:g}" for m in mu_arr])
        if k == 0:
            ax.set_yticks(range(n_phi))
            ax.set_yticklabels([f"{ph:.2f}" for ph in phi_arr])
            ax.set_ylabel(f"p = {int(p_arr[ip])}\nphi")
        if row_idx == len(higher_p_indices) - 1:
            ax.set_xlabel("mu")
        ax.set_title(f"AR({int(p_arr[ip])}) - AR(1), w = {int(w_v)}")
        for i in range(n_phi):
            for j in range(n_mu):
                v = data[i, j]
                if not np.isfinite(v):
                    text = "-"
                    color = "black"
                else:
                    text = f"{v:+.2f}"
                    color = "black" if abs(v) < 0.7 * vlim else "white"
                ax.text(j, i, text, ha="center", va="center",
                         color=color, fontsize=7)
        fig.colorbar(im, ax=ax, label="delta A*")
fig.suptitle(
    f"Delta A*_R2 from AR(1) (positive = higher-order AR erodes later)",
    y=1.005,
)
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_06_delta_a_star_r2_from_ar1.png",
             dpi=150, bbox_inches="tight")
plt.show()

## Save numeric summary

4D arrays for both A* metrics (filtered by hit rate), the raw hit rate matrices, the parameter grids (including `forecast_p_grid`), the Figure C erosion-curve arrays, and the run inputs. Small npz, fully self-contained for replotting.

In [ ]:
np.savez(
    f"{data_dir}/phase_06_a_star_grid.npz",
    mu_grid=mu_arr,
    phi_grid=phi_arr,
    w_grid=w_arr,
    forecast_p_grid=p_arr,
    a_star_R2=a_star_R2,
    a_star_phi=a_star_phi,
    hit_rate_R2=hit_rate_R2,
    hit_rate_phi=hit_rate_phi,
    baseline_R2_grid=baseline_R2_grid,
    baseline_phi_grid=baseline_phi_grid,
    # Figure C: erosion curves at one paired-seed cell across AR orders.
    line_curve_p_grid=p_arr,
    line_curve_adoption_share=line_curve_adoption_share,
    line_curve_residual_r2=line_curve_residual_r2,
    line_curve_phi=line_curve_phi_trace,
    line_curve_mu=np.array(line_curve_mu),
    line_curve_phi_param=np.array(line_curve_phi),
    line_curve_w=np.array(line_curve_w),
    line_curve_seed=np.array(line_curve_seed),
    eval_window=np.array(eval_window),
    risk_scale=np.array(risk_scale),
    q_cap=np.array(q_cap),
    adoption_pi=np.array(adoption_pi),
    r2_threshold_factor=np.array(r2_threshold_factor),
    phi_threshold_factor=np.array(phi_threshold_factor),
    min_hit_rate=np.array(min_hit_rate),
    baseline_span=np.array(baseline_span),
    T=np.array(T),
    N=np.array(N),
    num_seeds=np.array(num_seeds_phase6),
    base_seed=np.array(base_seed),
)

## Done when

- The two AR(1) headline heatmaps populate the bulk of the (mu, phi) grid for each w; cells that fail the hit-rate filter print `hit X.X` instead of a number, so an honest reader can tell empty cells from low-A* cells.
- Larger mu generally makes adoption feedback bite earlier (A* moves to the left in each panel). The phi gradient is less mechanically monotone: stronger underlying predictability can raise the baseline R^2 and sometimes push the relative R^2 threshold to a higher adoption share, so the figures should be interpreted as the joint effect of signal strength and feedback strength, not as a simple one-dimensional monotone relation in every parameter.
- The effective-phi heatmap is the cleaner read of the adoption-feedback channel because it does not depend on the AR fit quality; the residual-R^2 heatmap is a forecasting metric and its monotonicity is muddied by the baseline-R^2 dependence on phi.
- The by_p heatmap grids reproduce the AR(1) row at the top and show qualitatively similar gradients at higher p, demonstrating that the headline mechanism is not specific to AR(1) and survives multi-lag forecast specifications.
- The delta-from-AR(1) heatmap is mostly close to zero across the (mu, phi) grid, with magnitudes inside the relative threshold; that is the cleanest single robustness summary.
- The paired-seed erosion figure shows the rolling residual R^2 collapsing as adoption rises at every AR order, with the AR(1) line being the headline and the higher-order lines tracking it closely.
- The npz output includes `forecast_p_grid` and 4D A* arrays plus the Figure C erosion-curve arrays, so a replot script can pick out any (p, w) slice or the representative erosion curve without rerunning the sweep.

Phase 7 keeps `p = 1` as the proposal default for its transaction-cost extension; the AR(p > 1) robustness here lives entirely inside phase 6.